# 11 · Old vs new segmenter on the test split

Week 9 Day 3 of NeuroLens. v1 (notebook 07) and v2 (fine-tuned on the corrected cases, notebook 10) run on the same
188 test cases, in one run. The test split is opened once here, for both models.

Rules fixed before the run (`neurolens_ml/compare_seg.py`):
- each model's small-ET rule is tuned on the 148 val cases neither model trained on;
- **keep v2 if its mean test Dice is at most 0.005 below v1** (the margin used in fine-tuning);
- paired per-case difference with a bootstrap 95% interval;
- subgroup: test cases with a true enhancing tumor under 5 cm³, the kind of case the corrections were about.

**Kaggle setup:** GPU T4 x2 (one is used), Internet on, inputs `dschettler8845/brats-2021-task1`, the output of
`07_segmentation` (`segmenter_segresnet.pt`) and the output of `10. feedback loop` (`segmenter_segresnet_fb2.pt`).
Run with *Save Version → Save & Run All* (about 20 min), then download everything from *Output*.

In [ ]:
import importlib.util
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sanjardevx/neurolens.git"
BRANCH = "develop"
OLD_CONFIG, NEW_CONFIG = "ml/configs/seg_segresnet.yaml", "ml/configs/seg_finetune.yaml"
REPO_DIR = Path(os.environ.get("NEUROLENS_REPO_DIR", "/tmp/neurolens"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
EXTRA_ARGS = os.environ.get("NEUROLENS_ARGS", "").split()  # only for local smoke tests (fake data, checkpoint, --smoke)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
# --no-deps: keep Kaggle's own torch and numpy; MONAI and nibabel need nothing else
missing = [p for p in ["neurolens_ml", "monai", "nibabel"] if importlib.util.find_spec(p) is None]
if "neurolens_ml" in missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR / "ml")], check=True)
if pkgs := [p for p in missing if p != "neurolens_ml"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *pkgs], check=True)

import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("repo commit:", commit, "| GPU:", gpu, "| CPUs:", os.cpu_count(), "| free disk in /tmp:", shutil.disk_usage("/tmp").free // 10**9, "GB")

In [ ]:
cmd = [sys.executable, "-m", "neurolens_ml.compare_seg", "--old-config", str(REPO_DIR / OLD_CONFIG),
       "--new-config", str(REPO_DIR / NEW_CONFIG), "--out", str(OUT_DIR), *EXTRA_ARGS]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "comparison failed, see the log above"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

m = json.loads((OUT_DIR / "metrics_seg_compare.json").read_text())
m["repo_commit"], m["gpu"] = commit, gpu
(OUT_DIR / "metrics_seg_compare.json").write_text(json.dumps(m, indent=2))
rows = {"v1 all test": m["old"]["test"], "v2 all test": m["new"]["test"],
        "v1 small ET": m["small_et"]["old"], "v2 small ET": m["small_et"]["new"]}
display(pd.DataFrame(rows).T)
print("paired, all:", m["paired_all"])
print("paired, small ET:", m["small_et"]["paired"])
print("decision:", m["rule"], "->", "keep", m["keep"])

In [ ]:
# Each dot is one test case: above the line, v2 is better
df = pd.read_csv(OUT_DIR / "compare_test_seg.csv")
small = df.true_et_voxels_old < 5000
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], color="#999", lw=1)
ax.scatter(df.mean_old[~small], df.mean_new[~small], s=14, label="ET ≥ 5 cm³")
ax.scatter(df.mean_old[small], df.mean_new[small], s=14, label="ET < 5 cm³")
ax.set(xlabel="v1 mean Dice", ylabel="v2 mean Dice", title="Test cases: v1 vs v2", xlim=(0, 1), ylim=(0, 1))
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "compare_test_seg.png", dpi=90)
plt.show()